In [ ]:
import warnings, platform
import numpy as np
import pandas as pd
import pyreadstat
import pingouin as pg
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.inspection import permutation_importance
from statsmodels.miscmodels.ordinal_model import OrderedModel
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings('ignore')
plt.rc('font', family={'Windows': 'Malgun Gothic', 'Darwin': 'AppleGothic'}.get(platform.system(), 'NanumGothic'))
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.max_columns', None)

In [ ]:
df, meta = pyreadstat.read_sav('ZA7650_v2-0-0.sav', apply_value_formats=False, user_missing=True)
raw = df[~df['AGE'].between(0, 17)].reset_index(drop=True)

In [ ]:
names = {'v26': 'risk_price', 'v27': 'risk_tax', 'v28': 'risk_living',
         'v58': 'perc_air', 'v60': 'perc_weather', 'v41': 'aware_temp',
         'v21': 'worry_jobs', 'v24': 'growth_needed',
         'v14': 'trust_parliament', 'v11': 'trust_univ', 'v13': 'trust_business'}
demo = ['TOPBOT', 'EDULEVEL', 'AGE', 'SEX']
num = list(names) + demo

fin = raw[['c_alphan', 'WEIGHT'] + num].copy()
fin[num] = fin[num].mask(fin[num] < 0)
rev = ['v26', 'v27', 'v28', 'v41', 'v21', 'v24']
fin[rev] = 6 - fin[rev]
fin = fin.rename(columns=names)
fin['female'] = fin['SEX'].map({1: 0, 2: 1})
fin['party'] = pd.cut(raw['PARTY_LR'], [0, 3, 5], labels=['lib', 'con']).astype(object).fillna('none')

In [ ]:
risk = ['risk_price', 'risk_tax', 'risk_living']
att = ['perc_air', 'perc_weather', 'aware_temp', 'worry_jobs', 'growth_needed']
trust = ['trust_parliament', 'trust_univ', 'trust_business']
ctrl = ['TOPBOT', 'EDULEVEL', 'AGE', 'female']
xs = att + trust + ctrl
feat = risk + att + trust

In [ ]:
lab = {'risk_price': '가격인상 감수', 'risk_tax': '세금인상 감수', 'risk_living': '생활수준 하락 감수',
       'perc_air': '대기오염 체감', 'perc_weather': '극단기상 체감', 'aware_temp': '기온상승 위험인식',
       'worry_jobs': '물가/일자리 우선', 'growth_needed': '경제성장 필요',
       'trust_parliament': '신뢰: 의회', 'trust_univ': '신뢰: 대학/연구기관', 'trust_business': '신뢰: 기업',
       'TOPBOT': '주관적 계층', 'EDULEVEL': '교육수준', 'AGE': '연령', 'SEX': '성별', 'female': '여성'}
by = dict(tuple(fin.groupby('c_alphan')))

In [ ]:
pd.crosstab(raw['c_alphan'], raw['DATEYR'])

In [ ]:
miss = (raw[num] < 0).groupby(raw['c_alphan']).mean().mul(100).rename(columns=names).rename(columns=lab)
plt.figure(figsize=(14, 6))
sns.heatmap(miss.T, annot=True, fmt='.1f', cmap='Reds')
plt.title('국가별 문항 결측률(%)')
plt.show()

In [ ]:
pd.DataFrame({'전체': fin.groupby('c_alphan').size(), '분석 N': fin.dropna(subset=risk + xs).groupby('c_alphan').size()})

In [ ]:
fin.groupby('c_alphan').apply(lambda g: pg.cronbach_alpha(g[risk])[0]).round(3)

In [ ]:
cols = risk + att + trust + ctrl
w = fin['WEIGHT']
cm = fin[cols].mul(w, axis=0).groupby(fin['c_alphan']).sum() / fin[cols].notna().mul(w, axis=0).groupby(fin['c_alphan']).sum()
cm.round(2)

In [ ]:
rank = cm.rank(ascending=False)
pd.DataFrame({'KR': cm.loc['KR'], '28개국 평균': cm.mean(), '28개국 SD': cm.std(), 'KR 순위': rank.loc['KR']}).rename(index=lab).round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 7), sharex=True)
for a, y in zip(axes, risk):
    s = cm[y].sort_values()
    a.barh(s.index, s, color=np.where(s.index == 'KR', 'tab:red', 'tab:gray'))
    a.axvline(3, color='black', lw=.8)
    a.set_title(lab[y])
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4), sharey=True)
for a, x in zip(axes, ['perc_air', 'aware_temp', 'trust_parliament', 'trust_univ']):
    a.scatter(cm[x], cm['risk_tax'])
    for c in cm.index:
        a.annotate(c, (cm.loc[c, x], cm.loc[c, 'risk_tax']), fontsize=8)
    a.set(xlabel=lab[x], ylabel=lab['risk_tax'])
plt.tight_layout()
plt.show()

In [ ]:
Z = pd.DataFrame(StandardScaler().fit_transform(cm[feat]), index=cm.index, columns=feat)
pca = PCA().fit(Z)
pd.Series(pca.explained_variance_ratio_.cumsum(), index=range(1, len(feat) + 1)).round(3)

In [ ]:
n_pc = 3
pcs = PCA(n_components=n_pc).fit(Z)
score = pd.DataFrame(pcs.transform(Z), index=Z.index, columns=[f'PC{i}' for i in range(1, n_pc + 1)])
pd.DataFrame(pcs.components_.T, index=feat, columns=score.columns).rename(index=lab).round(2)

In [ ]:
bic = pd.DataFrame({cov: [GaussianMixture(k, covariance_type=cov, n_init=10, random_state=0).fit(score).bic(score) for k in range(1, 7)]
                    for cov in ['spherical', 'diag', 'full']}, index=range(1, 7))
bic.round(1)

In [ ]:
k, cov = bic.loc[2:].stack().idxmin()
gm = GaussianMixture(k, covariance_type=cov, n_init=10, random_state=0).fit(score)
info = pd.DataFrame({'cluster': gm.predict(score) + 1, 'prob': gm.predict_proba(score).max(axis=1)}, index=Z.index)
k, cov

In [ ]:
L = linkage(Z, 'ward')
info['ward'] = fcluster(L, k, 'maxclust')
plt.figure(figsize=(12, 4))
dendrogram(L, labels=Z.index.tolist())
plt.show()

In [ ]:
pd.Series({'silhouette GMM': silhouette_score(Z, info['cluster']),
           'silhouette Ward': silhouette_score(Z, info['ward']),
           'ARI(GMM, Ward)': adjusted_rand_score(info['cluster'], info['ward'])}).round(3)

In [ ]:
pd.crosstab(info['cluster'], info['ward'])

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(score, x='PC1', y='PC2', hue=info['cluster'], palette='Set1', s=80)
for c in score.index:
    plt.annotate(c, (score.loc[c, 'PC1'], score.loc[c, 'PC2']), fontsize=9)
plt.show()

In [ ]:
order = info.sort_values(['cluster', 'prob'], ascending=[True, False]).index
plt.figure(figsize=(10, 9))
sns.heatmap(Z.loc[order].rename(columns=lab), annot=True, fmt='.1f', cmap='RdBu_r', center=0)
plt.yticks(rotation=0)
plt.show()

In [ ]:
cm[feat].groupby(info['cluster']).mean().T.rename(index=lab).round(2)

In [ ]:
info['d_centroid'] = np.linalg.norm(Z - Z.groupby(info['cluster']).transform('mean'), axis=1)
info['d_KR'] = np.linalg.norm(Z - Z.loc['KR'], axis=1)
info.sort_values(['cluster', 'd_centroid']).round(3)

In [ ]:
multi = info['cluster'].value_counts().loc[lambda s: s > 1].index
cand = info[info['cluster'].isin(multi) & (info['prob'] >= .9)]
medoid = cand.groupby('cluster')['d_centroid'].idxmin().tolist()
single = info.index[~info['cluster'].isin(multi)].tolist()
nearest = info['d_KR'].drop('KR').idxmin()
farthest = info['d_KR'].idxmax()

In [ ]:
major = info.groupby('cluster')['ward'].agg(lambda s: s.mode()[0])
boundary = info.index[info['ward'] != info['cluster'].map(major)].tolist()
uncertain = info.index[info['prob'] < .9].tolist()
info.loc[list(dict.fromkeys(boundary + uncertain))].round(3)

In [ ]:
sel = list(dict.fromkeys(['KR', nearest, farthest] + medoid + single + boundary))
pd.Series({'KR 최근접': nearest, 'KR 최원거리': farthest, '군집 대표': medoid, '이탈 사례': single,
           '경계 사례': boundary, '소속 불확실': uncertain, '선정': sel})

In [ ]:
def ologit(d, y, x=xs):
    return OrderedModel.from_formula(f'{y} ~ {" + ".join(x)}', d, distr='logit').fit(method='bfgs', disp=False)

In [ ]:
models = {(c, y): ologit(d, y) for c, d in by.items() for y in risk}
res = pd.concat({k: pd.DataFrame({'OR': np.exp(m.params[xs]), 'p': m.pvalues[xs]}) for k, m in models.items()},
                names=['country', 'y', 'var']).reset_index()
res['lnOR'] = np.log(res['OR'])
res['cluster'] = res['country'].map(info['cluster'])
res.head()

In [ ]:
for y in risk:
    o = res[res['y'] == y].pivot(index='country', columns='var', values='lnOR').loc[order, xs]
    p = res[res['y'] == y].pivot(index='country', columns='var', values='p').loc[order, xs]
    plt.figure(figsize=(11, 9))
    sns.heatmap(o.rename(columns=lab), annot=np.where(p < .05, '*', ''), fmt='', cmap='RdBu_r', center=0)
    plt.title(f'{lab[y]} (log OR, 군집 순)')
    plt.show()

In [ ]:
key = ['perc_air', 'aware_temp', 'worry_jobs', 'growth_needed'] + trust + ['EDULEVEL']
g = sns.catplot(res[res['var'].isin(key)], x='cluster', y='lnOR', col='var', row='y', kind='box',
                col_order=key, height=2.3, sharey=False)
g.refline(y=0)
plt.show()

In [ ]:
res[res['var'].isin(key) & res['cluster'].isin(multi)].groupby(['var', 'y']).apply(
    lambda g: stats.kruskal(*[s['lnOR'] for _, s in g.groupby('cluster')]).pvalue
).unstack()[risk].reindex(key).rename(index=lab, columns=lab).round(3)

In [ ]:
sig = res[res['p'] < .05].assign(dir=lambda d: np.where(d['OR'] > 1, '+', '-'))
pd.crosstab(sig['var'], [sig['y'], sig['dir']]).reindex(index=xs, columns=risk, level=0).rename(index=lab, columns=lab)

In [ ]:
sub = fin[fin['c_alphan'].isin(sel)]
cm.loc[sel, risk + xs].T.rename(index=lab).round(2)

In [ ]:
long = sub.melt(id_vars=['c_alphan', 'WEIGHT'], value_vars=risk, var_name='item').dropna()
dist = pd.crosstab([long['item'], long['c_alphan']], long['value'], values=long['WEIGHT'], aggfunc='sum', normalize='index') * 100
dist = dist.reindex(risk, level=0).reindex(sel, level=1)
ax = dist.rename(index=lab).plot.barh(stacked=True, colormap='RdYlGn', figsize=(9, 1 + .4 * len(dist)), width=.8)
ax.invert_yaxis()
ax.legend(title='응답', bbox_to_anchor=(1, 1))
plt.show()

In [ ]:
vif = {}
for c in sel:
    X = sm.add_constant(by[c][xs].dropna())
    vif[c] = [variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])]
pd.DataFrame(vif, index=xs).rename(index=lab).round(2)

In [ ]:
ra = res[res['country'].isin(sel)]
grid = pd.MultiIndex.from_product([sel, risk])
ra.pivot_table(index='var', columns=['country', 'y'], values='OR').reindex(index=xs, columns=grid).rename(index=lab, columns=lab).round(3)

In [ ]:
ra.pivot_table(index='var', columns=['country', 'y'], values='p').reindex(index=xs, columns=grid).rename(index=lab, columns=lab).round(3)

In [ ]:
pd.DataFrame({k: {'N': m.nobs, 'AIC': m.aic, 'pseudo R2': m.prsquared} for k, m in models.items() if k[0] in sel})[grid].rename(columns=lab).round(3)

In [ ]:
for y in risk:
    g = by['KR'].dropna(subset=[y] + xs)
    X = sm.add_constant(g[xs])
    cut = {f'>={j}': np.exp(sm.Logit((g[y] >= j).astype(int), X).fit(disp=0).params[xs]) for j in range(2, 6)}
    display(pd.DataFrame(cut).assign(ologit=np.exp(models['KR', y].params[xs])).rename(index=lab).round(2))

In [ ]:
core = ['perc_air', 'aware_temp', 'worry_jobs'] + trust
inter = f"C(c_alphan, Treatment('KR')) * ({' + '.join(core)})"
pm = {y: ologit(sub, y, xs + [inter]) for y in risk}
pm0 = {y: ologit(sub, y, xs + ["C(c_alphan, Treatment('KR'))"]) for y in risk}
lr = pd.Series({y: 2 * (pm[y].llf - pm0[y].llf) for y in risk})
dfree = pd.Series({y: len(pm[y].params) - len(pm0[y].params) for y in risk})
pd.DataFrame({'LR': lr, 'df': dfree, 'p': stats.chi2.sf(lr, dfree)}).rename(index=lab).round(4)

In [ ]:
it = pd.concat({y: pd.DataFrame({'OR': np.exp(m.params), 'p': m.pvalues}).filter(like=':', axis=0) for y, m in pm.items()}, axis=1)
it.index = it.index.str.replace("C(c_alphan, Treatment('KR'))[T.", '').str.replace(']', '')
it.rename(columns=lab).round(3)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4), sharey=True)
for a, x in zip(axes, ['aware_temp', 'worry_jobs', 'trust_parliament', 'trust_univ']):
    for c in sel:
        vals = np.linspace(by[c][x].min(), by[c][x].max(), 9)
        new = pd.DataFrame([by[c][xs].mean()] * len(vals)).assign(**{x: vals})
        a.plot(vals, np.asarray(models[c, 'risk_tax'].predict(new))[:, 3:].sum(axis=1), marker='o', label=c)
    a.set(title=lab[x], ylabel='P(세금인상 감수 = 4, 5)')
axes[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
pd.crosstab(fin['c_alphan'], fin['party'], normalize='index').loc[['KR', 'US']].mul(100).round(1)

In [ ]:
xs_c = xs + ["C(party, Treatment('con'))"]
mc = {(c, y): ologit(by[c], y, xs_c) for c in ['KR', 'US'] for y in risk}
pd.DataFrame({k: {'N A': models[k].nobs, 'N C': m.nobs, 'AIC A': models[k].aic, 'AIC C': m.aic} for k, m in mc.items()}).rename(columns=lab).round(1)

In [ ]:
pd.concat({k: pd.DataFrame({'A': np.exp(models[k].params[core]), 'C': np.exp(m.params[core]), 'p_C': m.pvalues[core]})
           for k, m in mc.items()}, axis=1).rename(index=lab, columns=lab).round(3)

In [ ]:
pc = pd.concat({k: pd.DataFrame({'OR': np.exp(m.params), 'p': m.pvalues}).filter(like='party', axis=0) for k, m in mc.items()}, axis=1)
pc.index = pc.index.str.replace("C(party, Treatment('con'))[T.", '').str.replace(']', '')
pc.rename(columns=lab).round(3)

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=0)
lr_m = LogisticRegression(max_iter=2000)
rf_m = RandomForestClassifier(n_estimators=200, min_samples_leaf=10, random_state=0, n_jobs=-1)

In [ ]:
auc = {}
for c, y in models:
    d = by[c].dropna(subset=[y] + xs)
    X, t = d[xs], d[y] >= 4
    auc[c, y] = {'logit': cross_val_score(lr_m, X, t, cv=cv, scoring='roc_auc').mean(),
                 'RF': cross_val_score(rf_m, X, t, cv=cv, scoring='roc_auc').mean()}
auc = pd.DataFrame(auc).T.rename_axis(['country', 'y'])
auc['diff'] = auc['RF'] - auc['logit']

In [ ]:
auc.loc[sel].unstack('y').reindex(columns=risk, level=1).rename(columns=lab).round(3)

In [ ]:
auc.groupby(level='y').apply(lambda g: pd.Series({'logit': g['logit'].mean(), 'RF': g['RF'].mean(),
                                                  'p': stats.wilcoxon(g['RF'], g['logit']).pvalue})).loc[risk].rename(index=lab).round(3)

In [ ]:
auc.groupby(level='y').rank(ascending=False).xs('KR', level='country').loc[risk].rename(index=lab)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True)
for a, y in zip(axes, risk):
    s = auc.xs(y, level='y')
    a.scatter(s['logit'], s['RF'], c=np.where(s.index.isin(sel), 'tab:red', 'tab:gray'))
    for c in s.index:
        a.annotate(c, (s.loc[c, 'logit'], s.loc[c, 'RF']), fontsize=8)
    a.axline((.6, .6), slope=1, color='black', lw=.8)
    a.set(title=lab[y], xlabel='logit AUC (5-fold CV)', ylabel='RF AUC (5-fold CV)')
plt.tight_layout()
plt.show()

In [ ]:
imp = {}
for c, y in grid:
    d = by[c].dropna(subset=[y] + xs)
    Xtr, Xte, ttr, tte = train_test_split(d[xs], d[y] >= 4, test_size=.3, stratify=d[y] >= 4, random_state=0)
    imp[c, y] = permutation_importance(rf_m.fit(Xtr, ttr), Xte, tte, scoring='roc_auc', n_repeats=10, random_state=0).importances_mean
imp = pd.DataFrame(imp, index=xs)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=True)
for a, y in zip(axes, risk):
    sns.heatmap(imp.xs(y, axis=1, level=1).rename(index=lab), annot=True, fmt='.3f', cmap='RdBu_r', center=0, ax=a)
    a.set_title(lab[y])
plt.tight_layout()
plt.show()

In [ ]:
imp.rank(ascending=False).loc[trust].rename(index=lab, columns=lab).astype(int)